# ShadowFox Data Analyst Internship — Intermediate Level

## Customer & Revenue Performance Analysis

This notebook performs data cleaning validation, exploratory analysis, customer behaviour analysis, product/revenue analysis, segment and regional analysis, trend analysis, and business recommendations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('dataset.csv')
df['Order Date'] = pd.to_datetime(df['Order Date'])
df.head()

## 1. Data Understanding and Quality Checks

In [ ]:
print('Rows:', len(df))
print('Columns:', len(df.columns))
print('\nMissing values:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nData types:')
print(df.dtypes)

## 2. Key Business Metrics

The main metrics are total revenue, profit, orders, unique customers, repeat-customer rate, and profit margin.

In [ ]:
total_revenue = df['Revenue'].sum()
total_profit = df['Profit'].sum()
total_orders = df['Order ID'].nunique()
unique_customers = df['Customer ID'].nunique()
repeat_customers = (df.groupby('Customer ID')['Order ID'].nunique() > 1).sum()
repeat_rate = repeat_customers / unique_customers
profit_margin = total_profit / total_revenue

metrics = pd.Series({
    'Total Revenue': total_revenue,
    'Total Profit': total_profit,
    'Orders': total_orders,
    'Unique Customers': unique_customers,
    'Repeat Customer Rate': repeat_rate,
    'Profit Margin': profit_margin
})
metrics

## 3. Monthly Trend Analysis

In [ ]:
monthly = df.groupby(df['Order Date'].dt.to_period('M')).agg(
    Orders=('Order ID','nunique'),
    Revenue=('Revenue','sum'),
    Profit=('Profit','sum')
).reset_index()
monthly['Order Date'] = monthly['Order Date'].astype(str)
monthly

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(monthly['Order Date'], monthly['Revenue'], marker='o')
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Customer Behaviour and Repeat Purchases

In [ ]:
customer_summary = df.groupby(['Customer ID','Segment','Region','Acquisition Channel']).agg(
    Orders=('Order ID','nunique'),
    Revenue=('Revenue','sum'),
    Profit=('Profit','sum')
).reset_index()
customer_summary['Repeat Customer'] = np.where(customer_summary['Orders'] > 1, 'Yes', 'No')
customer_summary.sort_values('Revenue', ascending=False).head(10)

## 5. Product and Segment Performance

In [ ]:
product_summary = df.groupby('Product').agg(
    Units=('Quantity','sum'),
    Revenue=('Revenue','sum'),
    Profit=('Profit','sum')
).sort_values('Revenue', ascending=False)

segment_summary = df.groupby('Segment').agg(
    Revenue=('Revenue','sum'),
    Profit=('Profit','sum'),
    Orders=('Order ID','nunique')
).sort_values('Revenue', ascending=False)

display(product_summary)
display(segment_summary)

## 6. Regional and Acquisition-Channel Analysis

In [ ]:
region_summary = df.groupby('Region').agg(
    Revenue=('Revenue','sum'),
    Profit=('Profit','sum'),
    Orders=('Order ID','nunique')
).sort_values('Revenue', ascending=False)

channel_summary = df.groupby('Acquisition Channel').agg(
    Revenue=('Revenue','sum'),
    Profit=('Profit','sum'),
    Orders=('Order ID','nunique'),
    Customers=('Customer ID','nunique')
).sort_values('Revenue', ascending=False)

display(region_summary)
display(channel_summary)

## 7. Business Insights and Recommendations

- Identify revenue concentration by segment, product and region.
- Compare repeat and one-time customer behaviour.
- Investigate high- and low-performing months for seasonality or campaign effects.
- Compare acquisition channels using revenue and revenue per customer.
- Use the observed patterns to propose targeted actions, while treating them as analytical signals rather than proof of causation.